In [5]:
import pandas as pd
import numpy as np
from scipy import sparse
from recovery_model.system_builder import System
from scipy.sparse import coo_array, coo_matrix, linalg
import warnings
from itertools import chain, product

In [6]:
composition_dct = {
    "url": "data/Toy_WEEE_simplified/Toy_WEEE_Composition.xlsx",
    "sheet": "Toy_WEEE_composition",
    "mapper": {
        "Flow": "flows",
        "Layer1": "products",
        "Layer2": "components",
        "Layer3": "materials",
        "Layer4": "elements",
        "Value": "data",
        "Year": "year",
        "parameterCode": "layer_code",
    },
    "data_processing": {
        "components": ("c-p",),
        "materials": ("m-p", "m-c"),
        "elements": ("e-p", "e-c", "e-m"),
    },
}

tc_dct = {
    "url": "data/Toy_WEEE_simplified/Toy_WEEE_TCs.xlsx",
    "sheet": "Toy_WEEE_TCs",
    "mapper": {
        # inflows / outflows
        "input_flow": "inflows",
        "output_flow": "outflows",
        # levels
        "input_layer": "input_layer1_level",
        "input_sub_layer": "input_layer2_level",
        "output_layer": "output_layer1_level",
        # keys
        "input_layer_key": "input_layer1_key",
        "input_sub_layer_key": "input_layer2_key",
        "output_target_key": "output_layer1_key",
        # other columns
        "value": "data",
        "Year": "year",
    },
    "all_symbol": {
        "products": "P*",
        "components": "C*",
        "materials": "M*",
        "elements": "E*",
    },
    "crossboundary_inflows": ("WEEE_INFLOW",),
}

inputs_dct = {
    "url": "data/Toy_WEEE_simplified/Toy_WEEE_inputs.xlsx",
    "sheet": "Toy_WEEE_inputs",
    "unit": "tonnes",
    "mapper": {
        "input_flow": "flows",
        "Category": "products",
        "mass_(tonnes)": "data",
        "Year": "year",
    },
}

In [8]:
weee = System(composition_dct=composition_dct, inputs_dct=inputs_dct, tc_dct=tc_dct)

0
1
2
3
4
5
6


In [9]:
weee.index.shape

(486,)

In [10]:
weee.lneqs

<486x486 sparse matrix of type '<class 'numpy.float64'>'
	with 514 stored elements in Compressed Sparse Row format>

In [11]:
A = pd.DataFrame.sparse.from_spmatrix(weee.lneqs, index=weee.index, columns=weee.index)
A

flows                                          F1                            
products                                        ∅                            
components                                      ∅                            
materials                                       ∅             M1             
elements                                        ∅   E1   E2    ∅   E1   E2   
flows products components materials elements                                 
F1    ∅        ∅          ∅         ∅         1.0  0.0  0.0  0.0  0.0  0.0  \
                                    E1        0.0  1.0  0.0  0.0  0.0  0.0   
                                    E2        0.0  0.0  1.0  0.0  0.0  0.0   
                          M1        ∅         0.0  0.0  0.0  1.0  0.0  0.0   
                                    E1        0.0  0.0  0.0  0.0  1.0  0.0   
...                                           ...  ...  ...  ...  ...  ...   
F6    P2       C2         M1        E1        0.0  0.0  0.0  0.0  0.0  0.0   
                                    E2        0.0  0.0  0.0  0.0  0.0  0.0   
                          M2        ∅         0.0  0.0  0.0  0.0  0.0  0.0   
                                    E1        0.0  0.0  0.0  0.0  0.0  0.0   
                                    E2        0.0  0.0  0.0  0.0  0.0  0.0   

flows                                                             ...   F6   
products                                                          ...   P2   
components                                                    C1  ...   C1   
materials                                      M2              ∅  ...   M2   
elements                                        ∅   E1   E2    ∅  ...   E2   
flows products components materials elements                      ...        
F1    ∅        ∅          ∅         ∅         0.0  0.0  0.0  0.0  ...  0.0  \
                                    E1        0.0  0.0  0.0  0.0  ...  0.0   
                                    E2        0.0  0.0  0.0  0.0  ...  0.0   
                          M1        ∅         0.0  0.0  0.0  0.0  ...  0.0   
                                    E1        0.0  0.0  0.0  0.0  ...  0.0   
...                                           ...  ...  ...  ...  ...  ...   
F6    P2       C2         M1        E1        0.0  0.0  0.0  0.0  ...  0.0   
                                    E2        0.0  0.0  0.0  0.0  ...  0.0   
                          M2        ∅         0.0  0.0  0.0  0.0  ...  0.0   
                                    E1        0.0  0.0  0.0  0.0  ...  0.0   
                                    E2        0.0  0.0  0.0  0.0  ...  0.0   

flows                                                                        
products                                                                     
components                                     C2                            
materials                                       ∅             M1             
elements                                        ∅   E1   E2    ∅   E1   E2   
flows products components materials elements                                 
F1    ∅        ∅          ∅         ∅         0.0  0.0  0.0  0.0  0.0  0.0  \
                                    E1        0.0  0.0  0.0  0.0  0.0  0.0   
                                    E2        0.0  0.0  0.0  0.0  0.0  0.0   
                          M1        ∅         0.0  0.0  0.0  0.0  0.0  0.0   
                                    E1        0.0  0.0  0.0  0.0  0.0  0.0   
...                                           ...  ...  ...  ...  ...  ...   
F6    P2       C2         M1        E1        0.0  0.0  0.0  0.0  1.0  0.0   
                                    E2        0.0  0.0  0.0  0.0  0.0  1.0   
                          M2        ∅         0.0  0.0  0.0  0.0  0.0  0.0   
                                    E1        0.0  0.0  0.0  0.0  0.0  0.0   
                                    E2        0.0  0.0  0.0  0.0  0.0  0.0   

flows              

In [12]:
Y = pd.DataFrame.sparse.from_spmatrix(weee.y, index=weee.index).squeeze()
Y

flows  products  components  materials  elements
F1     ∅         ∅           ∅          ∅           0
                                        E1          0
                                        E2          0
                             M1         ∅           0
                                        E1          0
                                                   ..
F6     P2        C2          M1         E1          0
                                        E2          0
                             M2         ∅           0
                                        E1          0
                                        E2          0
Name: 0, Length: 486, dtype: Sparse[int64, 0]

In [14]:
solution = weee.solve()
solution

flows  products  components  materials  elements
F1     ∅         ∅           ∅          ∅           0.0
                                        E1          0.0
                                        E2          0.0
                             M1         ∅           0.0
                                        E1          0.0
                                                   ... 
F6     P2        C2          M1         E1          0.0
                                        E2          0.0
                             M2         ∅           0.0
                                        E1          0.0
                                        E2          0.0
Length: 486, dtype: float64

In [18]:
solution[solution != 0]

flows  products  components  materials  elements
F1     P1        ∅           ∅          ∅           1000.000000
                 C1          ∅          ∅           1355.932203
                             M1         ∅           1043.024772
                                        E1           521.512386
                                        E2           521.512386
                             M2         ∅            630.666141
                                        E1           567.599527
                                        E2            63.066614
                 C2          ∅          ∅            215.053763
                             M1         ∅             55.141991
                                        E1            33.085194
                                        E2            22.056796
                             M2         ∅            200.050013
                                        E1           140.035009
                                        E2            6

---


In [36]:
layer_lvls = ["input_layer1_level", "input_layer2_level", "output_layer1_level"]
layer_keys = ["input_layer1_key", "input_layer2_key", "output_layer1_key"]

filepath = tc_dct["url"]
sheet = tc_dct["sheet"]
df = pd.read_excel(filepath, sheet_name=sheet)
mapper = tc_dct["mapper"]
df = df.rename(mapper=mapper, axis=1)

In [37]:
layer_lvls = ["input_layer1_level", "input_layer2_level", "output_layer1_level"]
layer_keys = ["input_layer1_key", "input_layer2_key", "output_layer1_key"]

filepath = tc_dct["url"]
sheet = tc_dct["sheet"]
df = pd.read_excel(filepath, sheet_name=sheet)
mapper = tc_dct["mapper"]
df = df.rename(mapper=mapper, axis=1)
df

,inflows,input_layer1_level,input_layer1_key,input_layer2_level,input_layer2_key,outflows,output_layer1_level,output_layer1_key,process,technology,distribution_coeff,transformative_coeff,data
0,F0,products,Cat_1,NaN,NaN,F1,products,Cat_1,Process_1,NaN,NaN,NaN,0.5
1,F0,products,Cat_2,NaN,NaN,F1,products,Cat_2,Process_1,NaN,NaN,NaN,0.8
2,F0,products,Cat_1,NaN,NaN,F2,products,Cat_1,Process_1,NaN,NaN,NaN,0.5
3,F0,products,Cat_2,NaN,NaN,F2,products,Cat_2,Process_1,NaN,NaN,NaN,0.2
4,F1,materials,M1,NaN,NaN,F3,materials,M1,Process_2,NaN,NaN,NaN,0.8
5,F1,materials,M2,NaN,NaN,F4,materials,M2,Process_2,NaN,NaN,NaN,0.4
6,F2,materials,M1,NaN,NaN,F5,materials,M1,Process_3,NaN,NaN,NaN,0.6
7,F2,materials,M2,NaN,NaN,F6,materials,M2,Process_3,NaN,NaN,NaN,0.6
8,F1,products,P*,NaN,NaN,F7,materials,M1,Process_4,NaN,NaN,NaN,0.1
9,F1,products,Cat_1,NaN,NaN,F7,materials,M1,Process_4,NaN,NaN,NaN,0.2


In [38]:
# Sanity check
unique = [df[col].unique() for col in layer_lvls]
levels = set(chain.from_iterable(unique)) - {np.nan}
if not levels.issubset(set(weee.layer_names)):
    warnings.warn("Layers in TC file are not consistent with the system layers")

In [39]:
df_inflows = df[[layer_lvls[0], layer_keys[0]]].pivot(columns=layer_lvls[0], values=layer_keys[0])
df_inflows.columns.name = None
df_inflows

,components,materials,products
0,NaN,NaN,Cat_1
1,NaN,NaN,Cat_2
2,NaN,NaN,Cat_1
3,NaN,NaN,Cat_2
4,NaN,M1,NaN
5,NaN,M2,NaN
6,NaN,M1,NaN
7,NaN,M2,NaN
8,NaN,NaN,P*
9,NaN,NaN,Cat_1


In [40]:
df_inflows = weee.autocomplete(df_inflows, np.nan)
df_inflows["flows"] = df["inflows"]
df_inflows

,flows,products,components,materials,elements
0,F0,Cat_1,NaN,NaN,NaN
1,F0,Cat_2,NaN,NaN,NaN
2,F0,Cat_1,NaN,NaN,NaN
3,F0,Cat_2,NaN,NaN,NaN
4,F1,NaN,NaN,M1,NaN
5,F1,NaN,NaN,M2,NaN
6,F2,NaN,NaN,M1,NaN
7,F2,NaN,NaN,M2,NaN
8,F1,P*,NaN,NaN,NaN
9,F1,Cat_1,NaN,NaN,NaN


In [41]:
df_inflows2 = df[[layer_lvls[1], layer_keys[1]]].pivot(columns=layer_lvls[1], values=layer_keys[1])
df_inflows2.columns.name = None

df_inflows.update(df_inflows2, overwrite=False)
df_inflows

,flows,products,components,materials,elements
0,F0,Cat_1,NaN,NaN,NaN
1,F0,Cat_2,NaN,NaN,NaN
2,F0,Cat_1,NaN,NaN,NaN
3,F0,Cat_2,NaN,NaN,NaN
4,F1,NaN,NaN,M1,NaN
5,F1,NaN,NaN,M2,NaN
6,F2,NaN,NaN,M1,NaN
7,F2,NaN,NaN,M2,NaN
8,F1,P*,NaN,NaN,NaN
9,F1,Cat_1,NaN,NaN,NaN


In [42]:
df_outflows = df[[layer_lvls[2], layer_keys[2]]].pivot(columns=layer_lvls[2], values=layer_keys[2])
df_outflows.columns.name = None
df_outflows = weee.autocomplete(df_outflows, np.nan)
df_outflows["flows"] = df["outflows"]
df_outflows

,flows,products,components,materials,elements
0,F1,Cat_1,NaN,NaN,NaN
1,F1,Cat_2,NaN,NaN,NaN
2,F2,Cat_1,NaN,NaN,NaN
3,F2,Cat_2,NaN,NaN,NaN
4,F3,NaN,NaN,M1,NaN
5,F4,NaN,NaN,M2,NaN
6,F5,NaN,NaN,M1,NaN
7,F6,NaN,NaN,M2,NaN
8,F7,NaN,NaN,M1,NaN
9,F7,NaN,NaN,M1,NaN


In [43]:
# filter_func = lambda s: np.isin(s, set(tc_dct["all_symbol"].values()))
df_inflows.update(df_outflows, overwrite=False)
df_inflows

,flows,products,components,materials,elements
0,F0,Cat_1,NaN,NaN,NaN
1,F0,Cat_2,NaN,NaN,NaN
2,F0,Cat_1,NaN,NaN,NaN
3,F0,Cat_2,NaN,NaN,NaN
4,F1,NaN,NaN,M1,NaN
5,F1,NaN,NaN,M2,NaN
6,F2,NaN,NaN,M1,NaN
7,F2,NaN,NaN,M2,NaN
8,F1,P*,NaN,M1,NaN
9,F1,Cat_1,NaN,M1,NaN


In [44]:
for col in weee.layer_names[1:]:
    mask = df_inflows[col].eq(tc_dct["all_symbol"][col]) & df_outflows[col].notna()
    df_inflows.loc[mask, col] = df_outflows.loc[mask, col]
df_inflows

,flows,products,components,materials,elements
0,F0,Cat_1,NaN,NaN,NaN
1,F0,Cat_2,NaN,NaN,NaN
2,F0,Cat_1,NaN,NaN,NaN
3,F0,Cat_2,NaN,NaN,NaN
4,F1,NaN,NaN,M1,NaN
5,F1,NaN,NaN,M2,NaN
6,F2,NaN,NaN,M1,NaN
7,F2,NaN,NaN,M2,NaN
8,F1,P*,NaN,M1,NaN
9,F1,Cat_1,NaN,M1,NaN


In [45]:
def fillna(df, fillna="\u2205"):
    temp = df.copy()
    mask = True
    for col in weee.layer_names[-1:0:-1]:
        mask = mask & df.loc[:, col].isna()
        temp.loc[mask, col] = fillna
    for col in weee.layer_names[1:]:
        temp[col] = temp[col].fillna(tc_dct["all_symbol"][col])
    return temp


# df_inflows = fillna(df_inflows)
# df_inflows

fillna(df_inflows)

,flows,products,components,materials,elements
0,F0,Cat_1,∅,∅,∅
1,F0,Cat_2,∅,∅,∅
2,F0,Cat_1,∅,∅,∅
3,F0,Cat_2,∅,∅,∅
4,F1,P*,C*,M1,∅
5,F1,P*,C*,M2,∅
6,F2,P*,C*,M1,∅
7,F2,P*,C*,M2,∅
8,F1,P*,C*,M1,∅
9,F1,Cat_1,C*,M1,∅


In [46]:
mask = True
fillna = "\u2205"
for col in weee.layer_names[-1:0:-1]:
    mask = mask & df_inflows.loc[:, col].isna()
    df_inflows.loc[mask, col] = fillna
for col in weee.layer_names[1:]:
    df_inflows[col] = df_inflows[col].fillna(tc_dct["all_symbol"][col])
df_inflows

,flows,products,components,materials,elements
0,F0,Cat_1,∅,∅,∅
1,F0,Cat_2,∅,∅,∅
2,F0,Cat_1,∅,∅,∅
3,F0,Cat_2,∅,∅,∅
4,F1,P*,C*,M1,∅
5,F1,P*,C*,M2,∅
6,F2,P*,C*,M1,∅
7,F2,P*,C*,M2,∅
8,F1,P*,C*,M1,∅
9,F1,Cat_1,C*,M1,∅


In [47]:
df_outflows = df_outflows.fillna("\u2205")
df_outflows

,flows,products,components,materials,elements
0,F1,Cat_1,∅,∅,∅
1,F1,Cat_2,∅,∅,∅
2,F2,Cat_1,∅,∅,∅
3,F2,Cat_2,∅,∅,∅
4,F3,∅,∅,M1,∅
5,F4,∅,∅,M2,∅
6,F5,∅,∅,M1,∅
7,F6,∅,∅,M2,∅
8,F7,∅,∅,M1,∅
9,F7,∅,∅,M1,∅


In [48]:
df.columns.difference(layer_lvls + layer_keys + ["inflows", "outflows"])

Index(['data', 'distribution_coeff', 'process', 'technology',
       'transformative_coeff'],
      dtype='object')

In [49]:
df = pd.concat(
    {
        "inflow": df_inflows,
        "outflow": df_outflows,
        "info": df[df.columns.difference(layer_lvls + layer_keys + ["inflows", "outflows"])],
    },
    axis=1,
)
df

inflow                                        outflow                       
    flows products components materials elements   flows products components   
0      F0    Cat_1          ∅         ∅        ∅      F1    Cat_1          ∅  \
1      F0    Cat_2          ∅         ∅        ∅      F1    Cat_2          ∅   
2      F0    Cat_1          ∅         ∅        ∅      F2    Cat_1          ∅   
3      F0    Cat_2          ∅         ∅        ∅      F2    Cat_2          ∅   
4      F1       P*         C*        M1        ∅      F3        ∅          ∅   
5      F1       P*         C*        M2        ∅      F4        ∅          ∅   
6      F2       P*         C*        M1        ∅      F5        ∅          ∅   
7      F2       P*         C*        M2        ∅      F6        ∅          ∅   
8      F1       P*         C*        M1        ∅      F7        ∅          ∅   
9      F1    Cat_1         C*        M1        ∅      F7        ∅          ∅   
10     F1       P*         C*        M1        ∅      F7        ∅          ∅   
11     F1       P*         C1        M1        ∅      F7        ∅          ∅   
12     F1       P*         C*        M1        ∅      F7        ∅          ∅   
13     F1    Cat_1         C1        M1        ∅      F7        ∅          ∅   
14     F1    Cat_1         C*        M1        ∅      F7        ∅          ∅   
15     F1       P*         C1        M1        ∅      F7        ∅          ∅   

                      info                                            
   materials elements data distribution_coeff    process technology   
0          ∅        ∅  0.5                NaN  Process_1        NaN  \
1          ∅        ∅  0.8                NaN  Process_1        NaN   
2          ∅        ∅  0.5                NaN  Process_1        NaN   
3          ∅        ∅  0.2                NaN  Process_1        NaN   
4         M1        ∅  0.8                NaN  Process_2        NaN   
5         M2        ∅  0.4                NaN  Process_2        NaN   
6         M1        ∅  0.6                NaN  Process_3        NaN   
7         M2        ∅  0.6                NaN  Process_3        NaN   
8         M1        ∅  0.1                NaN  Process_4        NaN   
9         M1        ∅  0.2                NaN  Process_4        NaN   
10        M1        ∅  0.3                NaN  Process_4        NaN   
11        M1        ∅  0.4                NaN  Process_4        NaN   
12        M1        ∅  0.5                NaN  Process_4        NaN   
13        M1        ∅  0.6                NaN  Process_4        NaN   
14        M1        ∅  0.7                NaN  Process_4        NaN   
15        M1        ∅  0.8                NaN  Process_4        NaN   

                         
   transformative_coeff  
0                   NaN  
1                   NaN  
2                   NaN  
3                   NaN  
4                   NaN  
5                   NaN  
6                   NaN  
7                   NaN  
8                   NaN  
9                   NaN  
10                  NaN  
11                  NaN  
12                  NaN  
13                  NaN  
14                  NaN  
15                  NaN

In [50]:
def compute_priority(serie, all_symbol, nan_symbol):
    res = np.full_like(serie, 2, dtype=int)
    res[serie.eq(nan_symbol)] = 0
    res[serie.eq(all_symbol)] = 1
    return res


priority = np.zeros(df.shape[0], dtype=int)

for i, layer in enumerate(weee.layer_names[1:]):
    priority += (
        compute_priority(
            serie=df["inflow", layer],
            all_symbol=tc_dct["all_symbol"][layer],
            nan_symbol="\u2205",
        )
        * 10**i
    )

df[("other", "priority")] = priority
df

inflow                                        outflow                       
    flows products components materials elements   flows products components   
0      F0    Cat_1          ∅         ∅        ∅      F1    Cat_1          ∅  \
1      F0    Cat_2          ∅         ∅        ∅      F1    Cat_2          ∅   
2      F0    Cat_1          ∅         ∅        ∅      F2    Cat_1          ∅   
3      F0    Cat_2          ∅         ∅        ∅      F2    Cat_2          ∅   
4      F1       P*         C*        M1        ∅      F3        ∅          ∅   
5      F1       P*         C*        M2        ∅      F4        ∅          ∅   
6      F2       P*         C*        M1        ∅      F5        ∅          ∅   
7      F2       P*         C*        M2        ∅      F6        ∅          ∅   
8      F1       P*         C*        M1        ∅      F7        ∅          ∅   
9      F1    Cat_1         C*        M1        ∅      F7        ∅          ∅   
10     F1       P*         C*        M1        ∅      F7        ∅          ∅   
11     F1       P*         C1        M1        ∅      F7        ∅          ∅   
12     F1       P*         C*        M1        ∅      F7        ∅          ∅   
13     F1    Cat_1         C1        M1        ∅      F7        ∅          ∅   
14     F1    Cat_1         C*        M1        ∅      F7        ∅          ∅   
15     F1       P*         C1        M1        ∅      F7        ∅          ∅   

                      info                                            
   materials elements data distribution_coeff    process technology   
0          ∅        ∅  0.5                NaN  Process_1        NaN  \
1          ∅        ∅  0.8                NaN  Process_1        NaN   
2          ∅        ∅  0.5                NaN  Process_1        NaN   
3          ∅        ∅  0.2                NaN  Process_1        NaN   
4         M1        ∅  0.8                NaN  Process_2        NaN   
5         M2        ∅  0.4                NaN  Process_2        NaN   
6         M1        ∅  0.6                NaN  Process_3        NaN   
7         M2        ∅  0.6                NaN  Process_3        NaN   
8         M1        ∅  0.1                NaN  Process_4        NaN   
9         M1        ∅  0.2                NaN  Process_4        NaN   
10        M1        ∅  0.3                NaN  Process_4        NaN   
11        M1        ∅  0.4                NaN  Process_4        NaN   
12        M1        ∅  0.5                NaN  Process_4        NaN   
13        M1        ∅  0.6                NaN  Process_4        NaN   
14        M1        ∅  0.7                NaN  Process_4        NaN   
15        M1        ∅  0.8                NaN  Process_4        NaN   

                           other  
   transformative_coeff priority  
0                   NaN        2  
1                   NaN        2  
2                   NaN        2  
3                   NaN        2  
4                   NaN      211  
5                   NaN      211  
6                   NaN      211  
7                   NaN      211  
8                   NaN      211  
9                   NaN      212  
10                  NaN      211  
11                  NaN      221  
12                  NaN      211  
13                  NaN      222  
14                  NaN      212  
15                  NaN      221

In [51]:
for layer in weee.layer_names[1:]:  # not flows
    all_symbol = tc_dct["all_symbol"][layer]
    mask = df[("inflow", layer)].eq(all_symbol)
    temp_data = [weee.var["index"][layer].keys()] * mask.sum()
    temp_idx = mask[mask].index
    new_values = pd.Series(temp_data, index=temp_idx)
    df.loc[mask, ("inflow", layer)] = new_values

df

inflow                                                    outflow            
    flows           products   components materials elements   flows products   
0      F0              Cat_1            ∅         ∅        ∅      F1    Cat_1  \
1      F0              Cat_2            ∅         ∅        ∅      F1    Cat_2   
2      F0              Cat_1            ∅         ∅        ∅      F2    Cat_1   
3      F0              Cat_2            ∅         ∅        ∅      F2    Cat_2   
4      F1  (∅, Cat_1, Cat_2)  (∅, C1, C2)        M1        ∅      F3        ∅   
5      F1  (∅, Cat_1, Cat_2)  (∅, C1, C2)        M2        ∅      F4        ∅   
6      F2  (∅, Cat_1, Cat_2)  (∅, C1, C2)        M1        ∅      F5        ∅   
7      F2  (∅, Cat_1, Cat_2)  (∅, C1, C2)        M2        ∅      F6        ∅   
8      F1  (∅, Cat_1, Cat_2)  (∅, C1, C2)        M1        ∅      F7        ∅   
9      F1              Cat_1  (∅, C1, C2)        M1        ∅      F7        ∅   
10     F1  (∅, Cat_1, Cat_2)  (∅, C1, C2)        M1        ∅      F7        ∅   
11     F1  (∅, Cat_1, Cat_2)           C1        M1        ∅      F7        ∅   
12     F1  (∅, Cat_1, Cat_2)  (∅, C1, C2)        M1        ∅      F7        ∅   
13     F1              Cat_1           C1        M1        ∅      F7        ∅   
14     F1              Cat_1  (∅, C1, C2)        M1        ∅      F7        ∅   
15     F1  (∅, Cat_1, Cat_2)           C1        M1        ∅      F7        ∅   

                                 info                                 
   components materials elements data distribution_coeff    process   
0           ∅         ∅        ∅  0.5                NaN  Process_1  \
1           ∅         ∅        ∅  0.8                NaN  Process_1   
2           ∅         ∅        ∅  0.5                NaN  Process_1   
3           ∅         ∅        ∅  0.2                NaN  Process_1   
4           ∅        M1        ∅  0.8                NaN  Process_2   
5           ∅        M2        ∅  0.4                NaN  Process_2   
6           ∅        M1        ∅  0.6                NaN  Process_3   
7           ∅        M2        ∅  0.6                NaN  Process_3   
8           ∅        M1        ∅  0.1                NaN  Process_4   
9           ∅        M1        ∅  0.2                NaN  Process_4   
10          ∅        M1        ∅  0.3                NaN  Process_4   
11          ∅        M1        ∅  0.4                NaN  Process_4   
12          ∅        M1        ∅  0.5                NaN  Process_4   
13          ∅        M1        ∅  0.6                NaN  Process_4   
14          ∅        M1        ∅  0.7                NaN  Process_4   
15          ∅        M1        ∅  0.8                NaN  Process_4   

                                      other  
   technology transformative_coeff priority  
0         NaN                  NaN        2  
1         NaN                  NaN        2  
2         NaN                  NaN        2  
3         NaN                  NaN        2  
4         NaN                  NaN      211  
5         NaN                  NaN      211  
6         NaN                  NaN      211  
7         NaN                  NaN      211  
8         NaN                  NaN      211  
9         NaN                  NaN      212  
10        NaN                  NaN      211  
11        NaN                  NaN      221  
12        NaN                  NaN      211  
13        NaN                  NaN      222  
14        NaN                  NaN      212  
15        NaN                  NaN      221

In [52]:
for layer in weee.layer_names[1:]:
    df = df.explode(("inflow", layer))
df

inflow                                        outflow                       
    flows products components materials elements   flows products components   
0      F0    Cat_1          ∅         ∅        ∅      F1    Cat_1          ∅  \
1      F0    Cat_2          ∅         ∅        ∅      F1    Cat_2          ∅   
2      F0    Cat_1          ∅         ∅        ∅      F2    Cat_1          ∅   
3      F0    Cat_2          ∅         ∅        ∅      F2    Cat_2          ∅   
4      F1        ∅          ∅        M1        ∅      F3        ∅          ∅   
..    ...      ...        ...       ...      ...     ...      ...        ...   
14     F1    Cat_1         C1        M1        ∅      F7        ∅          ∅   
14     F1    Cat_1         C2        M1        ∅      F7        ∅          ∅   
15     F1        ∅         C1        M1        ∅      F7        ∅          ∅   
15     F1    Cat_1         C1        M1        ∅      F7        ∅          ∅   
15     F1    Cat_2         C1        M1        ∅      F7        ∅          ∅   

                      info                                            
   materials elements data distribution_coeff    process technology   
0          ∅        ∅  0.5                NaN  Process_1        NaN  \
1          ∅        ∅  0.8                NaN  Process_1        NaN   
2          ∅        ∅  0.5                NaN  Process_1        NaN   
3          ∅        ∅  0.2                NaN  Process_1        NaN   
4         M1        ∅  0.8                NaN  Process_2        NaN   
..       ...      ...  ...                ...        ...        ...   
14        M1        ∅  0.7                NaN  Process_4        NaN   
14        M1        ∅  0.7                NaN  Process_4        NaN   
15        M1        ∅  0.8                NaN  Process_4        NaN   
15        M1        ∅  0.8                NaN  Process_4        NaN   
15        M1        ∅  0.8                NaN  Process_4        NaN   

                           other  
   transformative_coeff priority  
0                   NaN        2  
1                   NaN        2  
2                   NaN        2  
3                   NaN        2  
4                   NaN      211  
..                  ...      ...  
14                  NaN      212  
14                  NaN      212  
15                  NaN      221  
15                  NaN      221  
15                  NaN      221  

[80 rows x 16 columns]

In [53]:
for layer in weee.layer_names[1:]:
    mask = df[("outflow", layer)].eq(tc_dct["all_symbol"][layer])
    df.loc[mask, ("outflow", layer)] = df.loc[mask, ("inflow", layer)]
df

inflow                                        outflow                       
    flows products components materials elements   flows products components   
0      F0    Cat_1          ∅         ∅        ∅      F1    Cat_1          ∅  \
1      F0    Cat_2          ∅         ∅        ∅      F1    Cat_2          ∅   
2      F0    Cat_1          ∅         ∅        ∅      F2    Cat_1          ∅   
3      F0    Cat_2          ∅         ∅        ∅      F2    Cat_2          ∅   
4      F1        ∅          ∅        M1        ∅      F3        ∅          ∅   
..    ...      ...        ...       ...      ...     ...      ...        ...   
14     F1    Cat_1         C1        M1        ∅      F7        ∅          ∅   
14     F1    Cat_1         C2        M1        ∅      F7        ∅          ∅   
15     F1        ∅         C1        M1        ∅      F7        ∅          ∅   
15     F1    Cat_1         C1        M1        ∅      F7        ∅          ∅   
15     F1    Cat_2         C1        M1        ∅      F7        ∅          ∅   

                      info                                            
   materials elements data distribution_coeff    process technology   
0          ∅        ∅  0.5                NaN  Process_1        NaN  \
1          ∅        ∅  0.8                NaN  Process_1        NaN   
2          ∅        ∅  0.5                NaN  Process_1        NaN   
3          ∅        ∅  0.2                NaN  Process_1        NaN   
4         M1        ∅  0.8                NaN  Process_2        NaN   
..       ...      ...  ...                ...        ...        ...   
14        M1        ∅  0.7                NaN  Process_4        NaN   
14        M1        ∅  0.7                NaN  Process_4        NaN   
15        M1        ∅  0.8                NaN  Process_4        NaN   
15        M1        ∅  0.8                NaN  Process_4        NaN   
15        M1        ∅  0.8                NaN  Process_4        NaN   

                           other  
   transformative_coeff priority  
0                   NaN        2  
1                   NaN        2  
2                   NaN        2  
3                   NaN        2  
4                   NaN      211  
..                  ...      ...  
14                  NaN      212  
14                  NaN      212  
15                  NaN      221  
15                  NaN      221  
15                  NaN      221  

[80 rows x 16 columns]

In [54]:
priority = pd.IndexSlice[("other", "priority")]
priority

('other', 'priority')

In [55]:
# df.loc[:, ("inflow", "outflow")]

df = df.sort_values(by=priority, ascending=False)
df = df[~df.loc[:, ["inflow", "outflow"]].duplicated(keep="first")]
df

inflow                                        outflow                       
    flows products components materials elements   flows products components   
13     F1    Cat_1         C1        M1        ∅      F7        ∅          ∅  \
15     F1    Cat_2         C1        M1        ∅      F7        ∅          ∅   
11     F1        ∅         C1        M1        ∅      F7        ∅          ∅   
14     F1    Cat_1          ∅        M1        ∅      F7        ∅          ∅   
9      F1    Cat_1         C2        M1        ∅      F7        ∅          ∅   
8      F1    Cat_2         C2        M1        ∅      F7        ∅          ∅   
8      F1    Cat_2          ∅        M1        ∅      F7        ∅          ∅   
10     F1        ∅          ∅        M1        ∅      F7        ∅          ∅   
10     F1        ∅         C2        M1        ∅      F7        ∅          ∅   
5      F1    Cat_2         C1        M2        ∅      F4        ∅          ∅   
5      F1    Cat_2         C2        M2        ∅      F4        ∅          ∅   
5      F1    Cat_2          ∅        M2        ∅      F4        ∅          ∅   
5      F1    Cat_1         C2        M2        ∅      F4        ∅          ∅   
5      F1    Cat_1         C1        M2        ∅      F4        ∅          ∅   
5      F1    Cat_1          ∅        M2        ∅      F4        ∅          ∅   
5      F1        ∅         C2        M2        ∅      F4        ∅          ∅   
5      F1        ∅         C1        M2        ∅      F4        ∅          ∅   
5      F1        ∅          ∅        M2        ∅      F4        ∅          ∅   
4      F1    Cat_2         C2        M1        ∅      F3        ∅          ∅   
4      F1    Cat_2         C1        M1        ∅      F3        ∅          ∅   
4      F1    Cat_2          ∅        M1        ∅      F3        ∅          ∅   
4      F1    Cat_1         C2        M1        ∅      F3        ∅          ∅   
4      F1    Cat_1         C1        M1        ∅      F3        ∅          ∅   
4      F1    Cat_1          ∅        M1        ∅      F3        ∅          ∅   
4      F1        ∅         C2        M1        ∅      F3        ∅          ∅   
4      F1        ∅         C1        M1        ∅      F3        ∅          ∅   
7      F2    Cat_2         C2        M2        ∅      F6        ∅          ∅   
6      F2        ∅          ∅        M1        ∅      F5        ∅          ∅   
7      F2        ∅          ∅        M2        ∅      F6        ∅          ∅   
6      F2        ∅         C1        M1        ∅      F5        ∅          ∅   
7      F2    Cat_2         C1        M2        ∅      F6        ∅          ∅   
7      F2    Cat_2          ∅        M2        ∅      F6        ∅          ∅   
7      F2    Cat_1         C2        M2        ∅      F6        ∅          ∅   
7      F2    Cat_1         C1        M2        ∅      F6        ∅          ∅   
7      F2    Cat_1          ∅        M2        ∅      F6        ∅          ∅   
7      F2        ∅         C2        M2        ∅      F6        ∅          ∅   
7      F2        ∅         C1        M2        ∅      F6        ∅          ∅   
4      F1        ∅          ∅        M1        ∅      F3        ∅          ∅   
6      F2    Cat_2         C2        M1        ∅      F5        ∅          ∅   
6      F2    Cat_2         C1        M1        ∅      F5        ∅          ∅   
6      F2    Cat_2          ∅        M1        ∅      F5        ∅          ∅   
6      F2    Cat_1         C2        M1        ∅      F5        ∅          ∅   
6      F2    Cat_1         C1        M1        ∅      F5        ∅          ∅   
6      F2    Cat_1          ∅        M1        ∅      F5        ∅          ∅   
6      F2        ∅         C2        M1        ∅      F5        ∅          ∅   
2      F0    Cat_1          ∅         ∅        ∅      F2    Cat_1          ∅   
1      F0    Cat_2          ∅         ∅        ∅      F1    Cat_2          ∅   
3      F0    Cat_2          ∅         ∅        ∅      F2    Cat_2          ∅   
0  

In [60]:
df = df.sort_index()
df

inflow                                        outflow                       
    flows products components materials elements   flows products components   
0      F0    Cat_1          ∅         ∅        ∅      F1    Cat_1          ∅  \
1      F0    Cat_2          ∅         ∅        ∅      F1    Cat_2          ∅   
2      F0    Cat_1          ∅         ∅        ∅      F2    Cat_1          ∅   
3      F0    Cat_2          ∅         ∅        ∅      F2    Cat_2          ∅   
4      F1        ∅          ∅        M1        ∅      F3        ∅          ∅   
4      F1        ∅         C1        M1        ∅      F3        ∅          ∅   
4      F1    Cat_1          ∅        M1        ∅      F3        ∅          ∅   
4      F1    Cat_1         C1        M1        ∅      F3        ∅          ∅   
4      F1    Cat_1         C2        M1        ∅      F3        ∅          ∅   
4      F1    Cat_2          ∅        M1        ∅      F3        ∅          ∅   
4      F1    Cat_2         C1        M1        ∅      F3        ∅          ∅   
4      F1    Cat_2         C2        M1        ∅      F3        ∅          ∅   
4      F1        ∅         C2        M1        ∅      F3        ∅          ∅   
5      F1    Cat_1         C1        M2        ∅      F4        ∅          ∅   
5      F1    Cat_1          ∅        M2        ∅      F4        ∅          ∅   
5      F1        ∅         C2        M2        ∅      F4        ∅          ∅   
5      F1        ∅         C1        M2        ∅      F4        ∅          ∅   
5      F1        ∅          ∅        M2        ∅      F4        ∅          ∅   
5      F1    Cat_2          ∅        M2        ∅      F4        ∅          ∅   
5      F1    Cat_2         C2        M2        ∅      F4        ∅          ∅   
5      F1    Cat_2         C1        M2        ∅      F4        ∅          ∅   
5      F1    Cat_1         C2        M2        ∅      F4        ∅          ∅   
6      F2        ∅         C1        M1        ∅      F5        ∅          ∅   
6      F2    Cat_2         C2        M1        ∅      F5        ∅          ∅   
6      F2        ∅          ∅        M1        ∅      F5        ∅          ∅   
6      F2    Cat_2         C1        M1        ∅      F5        ∅          ∅   
6      F2    Cat_1         C1        M1        ∅      F5        ∅          ∅   
6      F2    Cat_1         C2        M1        ∅      F5        ∅          ∅   
6      F2    Cat_1          ∅        M1        ∅      F5        ∅          ∅   
6      F2        ∅         C2        M1        ∅      F5        ∅          ∅   
6      F2    Cat_2          ∅        M1        ∅      F5        ∅          ∅   
7      F2        ∅         C2        M2        ∅      F6        ∅          ∅   
7      F2        ∅         C1        M2        ∅      F6        ∅          ∅   
7      F2    Cat_1          ∅        M2        ∅      F6        ∅          ∅   
7      F2    Cat_2         C1        M2        ∅      F6        ∅          ∅   
7      F2    Cat_1         C2        M2        ∅      F6        ∅          ∅   
7      F2    Cat_2          ∅        M2        ∅      F6        ∅          ∅   
7      F2        ∅          ∅        M2        ∅      F6        ∅          ∅   
7      F2    Cat_2         C2        M2        ∅      F6        ∅          ∅   
7      F2    Cat_1         C1        M2        ∅      F6        ∅          ∅   
8      F1    Cat_2         C2        M1        ∅      F7        ∅          ∅   
8      F1    Cat_2          ∅        M1        ∅      F7        ∅          ∅   
9      F1    Cat_1         C2        M1        ∅      F7        ∅          ∅   
10     F1        ∅          ∅        M1        ∅      F7        ∅          ∅   
10     F1        ∅         C2        M1        ∅      F7        ∅          ∅   
11     F1        ∅         C1        M1        ∅      F7        ∅          ∅   
13     F1    Cat_1         C1        M1        ∅      F7        ∅          ∅   
14     F1    Cat_1          ∅        M1        ∅      F7        ∅          ∅   
15 

In [64]:
df[("info", "data")]

0     0.5
1     0.8
2     0.5
3     0.2
4     0.8
4     0.8
4     0.8
4     0.8
4     0.8
4     0.8
4     0.8
4     0.8
4     0.8
5     0.4
5     0.4
5     0.4
5     0.4
5     0.4
5     0.4
5     0.4
5     0.4
5     0.4
6     0.6
6     0.6
6     0.6
6     0.6
6     0.6
6     0.6
6     0.6
6     0.6
6     0.6
7     0.6
7     0.6
7     0.6
7     0.6
7     0.6
7     0.6
7     0.6
7     0.6
7     0.6
8     0.1
8     0.1
9     0.2
10    0.3
10    0.3
11    0.4
13    0.6
14    0.7
15    0.8
Name: (info, data), dtype: float64

---


In [123]:
# Sanity check that output is at a more granular level than the input (or same level)
temp_df = df[layer_lvls].copy()
for col in layer_lvls:
    # replace layer level with its index (products = 1, components = 2, ...)
    temp_df[col] = temp_df[col].map({v: k for k, v in enumerate(weee.layer_names)})

mask = (temp_df.iloc[:, 2] < temp_df.iloc[:, 0]) | (temp_df.iloc[:, 2] < temp_df.iloc[:, 1])

if mask.sum() > 0:
    raise ValueError(f"The output layer is broader than the input layers at rows: {mask[mask].index.values}")

In [124]:
# Sanity check on rows with "all" symbols
mask = df[layer_keys].isin(set(tc_dct["all_symbol"].values()))

# check if there's two "all" symbols on the input side
if (mask.iloc[:, :2].sum(axis=1) > 1).any():
    multiple_all_wth_inputs = mask.iloc[:, :2].sum(axis=1) > 1
    index = df[multiple_all_wth_inputs].index
    warnings.warn(
        f"There are two 'all' symbols on the input side, at rows: {index.values}.\n" "Make sure it is not a mistake."
    )

/tmp/ipykernel_1205870/1709501189.py:8: UserWarning: There are two 'all' symbols on the input side, at rows: [9].
Make sure it is not a mistake.
  warnings.warn(


In [125]:
mask = mask.any(axis=1)
temp_df = df.loc[mask, :].copy()
temp_df

,inflows,input_layer1_level,input_layer1_key,input_layer2_level,input_layer2_key,input_sub_layer_key.1,outflows,output_layer1_level,output_layer1_key,process,technology,distribution_coeff,transformative_coeff,data
8,F0,products,P*,NaN,NaN,NaN,F7,materials,M1,Process_4,NaN,NaN,NaN,0.95
9,F0,products,P*,materials,M*,NaN,F7,materials,M1,Process_4,NaN,NaN,NaN,0.95
10,F0,products,P*,NaN,NaN,NaN,F7,materials,M*,Process_4,NaN,NaN,NaN,0.95
11,F0,products,P*,NaN,NaN,NaN,F7,components,C*,Process_4,NaN,NaN,NaN,0.95


In [122]:
for layer in weee.layer_names[1:]:  # not flows
    all_symbol = tc_dct["all_symbol"][layer]
    for col in layer_keys:
        mask = temp_df[col].eq(all_symbol)
        temp_data = [weee.var["index"][layer].keys()] * mask.sum()
        temp_idx = mask[mask].index
        new_values = pd.Series(temp_data, index=temp_idx)
        temp_df.loc[mask, col] = new_values

temp_df

,inflows,input_layer1_level,input_layer1_key,input_layer2_level,input_layer2_key,input_sub_layer_key.1,outflows,output_layer1_level,output_layer1_key,process,technology,distribution_coeff,transformative_coeff,data
8,F0,products,"(∅, Cat_1, Cat_2)",NaN,NaN,NaN,F7,materials,M1,Process_4,NaN,NaN,NaN,0.95
9,F0,products,"(∅, Cat_1, Cat_2)",materials,"(∅, M1, M2)",NaN,F7,materials,M1,Process_4,NaN,NaN,NaN,0.95
10,F0,products,"(∅, Cat_1, Cat_2)",NaN,NaN,NaN,F7,materials,"(∅, M1, M2)",Process_4,NaN,NaN,NaN,0.95
11,F0,products,"(∅, Cat_1, Cat_2)",NaN,NaN,NaN,F7,components,"(∅, C1, C2)",Process_4,NaN,NaN,NaN,0.95


In [107]:
temp_df.explode("input_layer1_key").explode("input_layer2_key").explode("output_layer1_key")

,inflows,input_layer1_level,input_layer1_key,input_layer2_level,input_layer2_key,input_sub_layer_key.1,outflows,output_layer1_level,output_layer1_key,process,technology,distribution_coeff,transformative_coeff,data
8,F0,products,Cat_1,NaN,NaN,NaN,F7,materials,M1,Process_4,NaN,NaN,NaN,0.95
8,F0,products,Cat_2,NaN,NaN,NaN,F7,materials,M1,Process_4,NaN,NaN,NaN,0.95
9,F0,products,Cat_1,materials,M1,NaN,F7,materials,M1,Process_4,NaN,NaN,NaN,0.95
9,F0,products,Cat_1,materials,M2,NaN,F7,materials,M1,Process_4,NaN,NaN,NaN,0.95
9,F0,products,Cat_2,materials,M1,NaN,F7,materials,M1,Process_4,NaN,NaN,NaN,0.95
9,F0,products,Cat_2,materials,M2,NaN,F7,materials,M1,Process_4,NaN,NaN,NaN,0.95
10,F0,products,Cat_1,NaN,NaN,NaN,F7,materials,M1,Process_4,NaN,NaN,NaN,0.95
10,F0,products,Cat_1,NaN,NaN,NaN,F7,materials,M2,Process_4,NaN,NaN,NaN,0.95
10,F0,products,Cat_2,NaN,NaN,NaN,F7,materials,M1,Process_4,NaN,NaN,NaN,0.95
10,F0,products,Cat_2,NaN,NaN,NaN,F7,materials,M2,Process_4,NaN,NaN,NaN,0.95


dict_keys(['∅', 'Al', 'Cu'])

In [22]:
temp_df = temp_df.explode("input_layer1_key").explode("input_layer2_key").explode("output_layer1_key")
temp_df

,inflows,input_layer1_level,input_layer1_key,input_layer2_level,input_layer2_key,input_sub_layer_key.1,outflows,output_layer1_level,output_layer1_key,process,technology,distribution_coeff,transformative_coeff,data
8,F0,products,Cat_1,NaN,NaN,NaN,F7,materials,M1,Process_4,NaN,NaN,NaN,0.95
8,F0,products,Cat_2,NaN,NaN,NaN,F7,materials,M1,Process_4,NaN,NaN,NaN,0.95
9,F0,products,Cat_1,materials,M1,NaN,F7,materials,M1,Process_4,NaN,NaN,NaN,0.95
9,F0,products,Cat_1,materials,M2,NaN,F7,materials,M1,Process_4,NaN,NaN,NaN,0.95
9,F0,products,Cat_2,materials,M1,NaN,F7,materials,M1,Process_4,NaN,NaN,NaN,0.95
9,F0,products,Cat_2,materials,M2,NaN,F7,materials,M1,Process_4,NaN,NaN,NaN,0.95
10,F0,products,Cat_1,NaN,NaN,NaN,F7,materials,M1,Process_4,NaN,NaN,NaN,0.95
10,F0,products,Cat_1,NaN,NaN,NaN,F7,materials,M2,Process_4,NaN,NaN,NaN,0.95
10,F0,products,Cat_2,NaN,NaN,NaN,F7,materials,M1,Process_4,NaN,NaN,NaN,0.95
10,F0,products,Cat_2,NaN,NaN,NaN,F7,materials,M2,Process_4,NaN,NaN,NaN,0.95


In [25]:
temp_df["input_layer1_level"].map({v: k for k, v in enumerate(weee.layer_names)})

8     1
8     1
9     1
9     1
9     1
9     1
10    1
10    1
10    1
10    1
11    1
11    1
11    1
11    1
11    1
11    1
11    1
11    1
9     1
9     1
9     1
9     1
10    1
10    1
10    1
10    1
Name: input_layer1_level, dtype: int64

In [13]:
all_symbol = "P*"
cols_set = ["input_layer1_key", "input_layer2_key", "output_layer1_key"]
mask = df[cols_set].eq(all_symbol)
df[mask][cols_set].dropna(how="all").to_dict(orient="records")

[]

In [4]:
weee.input_rows

MultiIndex([('F0', 'Cat_1', '∅', '∅', '∅'),
            ('F0', 'Cat_2', '∅', '∅', '∅')],
           names=['flows', 'products', 'components', 'materials', 'elements'])

In [5]:
weee.index

MultiIndex([('F0',     '∅',  '∅',  '∅',  '∅'),
            ('F0',     '∅',  '∅',  '∅', 'Al'),
            ('F0',     '∅',  '∅',  '∅', 'Cu'),
            ('F0',     '∅',  '∅', 'M1',  '∅'),
            ('F0',     '∅',  '∅', 'M1', 'Al'),
            ('F0',     '∅',  '∅', 'M1', 'Cu'),
            ('F0',     '∅',  '∅', 'M2',  '∅'),
            ('F0',     '∅',  '∅', 'M2', 'Al'),
            ('F0',     '∅',  '∅', 'M2', 'Cu'),
            ('F0',     '∅', 'C1',  '∅',  '∅'),
            ...
            ('F6', 'Cat_2', 'C1', 'M2', 'Cu'),
            ('F6', 'Cat_2', 'C2',  '∅',  '∅'),
            ('F6', 'Cat_2', 'C2',  '∅', 'Al'),
            ('F6', 'Cat_2', 'C2',  '∅', 'Cu'),
            ('F6', 'Cat_2', 'C2', 'M1',  '∅'),
            ('F6', 'Cat_2', 'C2', 'M1', 'Al'),
            ('F6', 'Cat_2', 'C2', 'M1', 'Cu'),
            ('F6', 'Cat_2', 'C2', 'M2',  '∅'),
            ('F6', 'Cat_2', 'C2', 'M2', 'Al'),
            ('F6', 'Cat_2', 'C2', 'M2', 'Cu')],
           names=['flows', 'products', 'com

In [8]:
weee.get_indexer(weee.input_rows)

array([27, 54])

In [9]:
weee.get_indexer(weee.comp_cols)

array([27, 27, 36, 36, 45, 45, 39, 39, 42, 42, 48, 48, 51, 51, 54, 54, 63,
       63, 72, 72, 66, 66, 69, 69, 75, 75, 78, 78])

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33,
       34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50,
       51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67,
       68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80])

In [10]:
ones = np.ones(weee.index.shape)
suite = np.arange(len(weee.index))
data = np.hstack((ones, -weee.comp_data))
rows = np.hstack((suite, weee.get_indexer(weee.comp_rows)))
cols = np.hstack((suite, weee.get_indexer(weee.comp_cols)))

coo_arr = coo_array((data, (rows, cols)), shape=(len(weee.index), len(weee.index)))
A = coo_arr.tocsc()

In [20]:
df_A = pd.DataFrame.sparse.from_spmatrix(A, index=weee.index, columns=weee.index)
df_A.loc[df_A.sum(axis=1) != 1, df_A.sum(axis=0) != 1].to_csv("results/composition_as_matrix.csv")

In [11]:
data = weee.input_data
rows = weee.get_indexer(weee.input_rows)

cols = np.zeros(rows.shape)
coo_arr = coo_array((data, (rows, cols)), shape=(len(weee.index), 1))
y = coo_arr.tocsc()

In [12]:
sol = linalg.spsolve(A, y)

In [13]:
pd.Series(sol, index=weee.index).to_csv("results/solved_composition.csv")

In [ ]:
tc_cols = weee.tc_cols.copy()
tc_cols

In [ ]:
tc_cols = tc_cols.to_frame()
tc_cols

In [ ]:
mask1 = 

In [ ]:
tc_cols = np.array(list(map(np.array, tc_cols.values)))
tc_cols

In [ ]:
tc_cols[:, 1]

In [ ]:
tc_rows = weee.tc_rows.copy()
tc_rows

In [ ]:
list(weee.tc_rows)

In [ ]:
list(weee.index[weee.tc_rows])
# list(weee.tc_rows)

In [ ]:
list(weee.index[weee.tc_cols])
# list(weee.tc_cols)

In [ ]:
pd.concat(
    {
        "inflow": weee.index[weee.tc_cols].to_frame().reset_index(drop=True),
        "outflow": weee.index[weee.tc_rows].to_frame().reset_index(drop=True),
        "TC": pd.DataFrame({"value": weee.tc_data}),
    },
    axis=1,
).to_csv("results/formatted_Toy_WEEE_TCs.csv", index=False)

In [ ]:
list(weee.index[weee.input_rows])

In [ ]:
list(weee.index[weee.comp_rows])

In [ ]:
print(weee)

In [ ]:
list(weee.index)

In [ ]:
weee.flows

In [ ]:
weee.products

In [ ]:
weee.materials

In [ ]:
weee.elements

In [ ]:
print(weee)

In [ ]:
idxs = [
    ["WEEE_INFLOW", slice(None), "Cables", -1, -1],
    pd.IndexSlice["WEEE_INFLOW", "Cat_1", "Cables", :, :],
]

for idx in idxs:
    print(weee.format_index(idx), "\n")

In [ ]:
idxs = [
    "WEEE_INFLOW",
    ["WEEE_INFLOW"],
    ("WEEE_INFLOW", "Cat_1"),
    ("WEEE_INFLOW", "Cat_1", "Cables", -1, -1),
    [("WEEE_INFLOW", "Cat_1", "Cables", -1, -1)],
    (["WEEE_INFLOW", "Cat_1", "Cables", -1, -1]),
    [["WEEE_INFLOW", "Cat_1", "Cables", -1, -1]],
    [
        ["WEEE_INFLOW", "Cat_1", "Cables", -1, -1],
        ["WEEE_INFLOW", "Cat_2", "Cables", -1, -1],
    ],
    ["WEEE_INFLOW", slice(None), "Cables", -1, -1],
    pd.IndexSlice["WEEE_INFLOW", "Cat_1", "Cables", :, :],
]

for idx in idxs:
    print(idx)
    print(weee.get_indexer(idx), "\n")

In [ ]:
int_idx = [240, 241, 242, 243, 244, 245]
list(weee.index_iloc(int_idx))

In [ ]:
weee.index.shape

In [ ]:
random_iloc = np.random.randint(0, weee.index.shape, 100)
random_idx = weee.index[random_iloc]
random_iloc

In [ ]:
%%timeit
weee.get_indexer(random_idx)

In [ ]:
%%timeit
weee.index.get_indexer(random_idx)

In [ ]:
%%timeit
weee.index.get_indexer(weee.format_index(random_idx))

In [ ]:
weee.format_index(random_idx)

In [ ]:
mi = pd.MultiIndex.from_arrays([list("abb"), list("def"), list("xyz")], names=["col1", "col2", "col3"])
mi

In [ ]:
keys = [1, 2]
mi.to_frame().iloc[keys].index

In [ ]:
mi[keys]

---


In [ ]:
solution = weee.solve()
solution.to_csv("results/solution.csv")
solution

In [ ]:
np.vstack(
    [
        weee.tcs.tocoo().row,
        weee.tcs.tocoo().col,
        # weee.tcs.tocoo().data,
    ]
)

In [ ]:
tcs = weee.tcs.copy()
temp_tc = pd.DataFrame(tcs.toarray(), index=weee.index, columns=weee.index)


xbrdy_inflow = tc_dct["crossboundary_inflows"][0]
# for cat in ("Cat_1", "Cat_2", "Cat_3", "Cat_4a", "Cat_4b", "Cat_5", "Cat_6"):
#     idx_ax0 = pd.IndexSlice[xbrdy_inflow, cat, -1, -1]
#     idx_ax1 = pd.IndexSlice["products", cat]
#     temp_tc.loc[idx_ax0, idx_ax1] = 1

idx = pd.IndexSlice[xbrdy_inflow, :, :, :]
temp_tc.loc[idx, idx].to_csv("results/temp_tc.csv")

In [ ]:
y = weee.y.copy()
temp_y = pd.DataFrame(y.toarray(), index=weee.index, columns=weee.columns)


temp_y.loc[idx].to_csv("results/temp_y.csv")

In [ ]:
(temp_tc * temp_y).loc[pd.IndexSlice[xbrdy_inflow, :, :, :]].to_csv("tc_dot_y.csv")

In [ ]:
A = temp_tc.loc[pd.IndexSlice[xbrdy_inflow, :, :, :]]
A.to_numpy().shape

In [ ]:
Y = temp_y.loc[pd.IndexSlice[xbrdy_inflow, :, :, :]]
Y.to_numpy().shape

In [ ]:
file = "data/test/TC Matrix Building v2.xlsx"
tcs = pd.read_excel(file, sheet_name="tc", header=None).values
y = pd.read_excel(file, sheet_name="y", header=None).values

In [ ]:
from scipy.sparse.linalg import spsolve
from scipy.sparse import csr_matrix, csc_matrix

A = csr_matrix(tcs)
Y = csr_matrix(y)
spsolve(A, Y)

In [ ]:
Afull = csr_matrix(tcs)
Afull.nonzero()

In [ ]:
np.vstack([Afull.nonzero(), Afull.data]).T

In [ ]:
def squarify(mat):
    max_dim = max(mat.shape)
    dims = np.array(mat.shape)
    new_shape = np.full(dims.shape, fill_value=max_dim)
    new_mat = np.zeros(new_shape)
    new_mat[: mat.shape[0], : mat.shape[1]] = mat
    return new_mat


new_A = squarify(A.to_numpy())

In [ ]:
res = new_A * Y.to_numpy()
pd.DataFrame(res, index=A.index, columns=Y.columns).to_csv("temp_res.csv")

In [ ]:
xbrdy_inflow = tc_dct["crossboundary_inflows"][0]
pp = pd.DataFrame(np.zeros(shape=(len(weee.index), len(weee.columns))), index=weee.index, columns=weee.columns)

prod = "Cat_1"
comp = slice(None)
mat = -1
idx_ax0 = pd.IndexSlice[xbrdy_inflow, prod, comp, mat]
idx_ax1 = pd.IndexSlice["products", prod]

pp.loc[idx_ax0, idx_ax1]

In [ ]:
def pruned(coo_mat, idxs, cols):
    csr_mat = coo_mat.tocsr()

    rows, _ = csr_mat.nonzero()
    unique_rows = np.sort(np.unique(rows))
    data = csr_mat[unique_rows, :].toarray()
    return pd.DataFrame(data=data, index=idxs[unique_rows], columns=cols)

In [ ]:
pruned(weee.y, weee.index, weee.columns).to_csv("test.csv")

In [ ]:
# i = weee.materials[0]
# i
rw = tuple([[-1, i, -1] for i in weee.materials])
cl = tuple([("materials", i) for i in weee.materials])
rw

<hr>

# Sparse matrices


In [ ]:
from scipy.sparse import coo_array, coo_matrix, linalg

In [ ]:
rows1 = np.random.randint(6, size=10)
cols1 = np.random.randint(6, size=10)
data1 = np.random.randint(10, size=10)

print(np.array([rows1, cols1, data1]))

coo_arr1 = sparse.coo_array((data1, (rows1, cols1)), shape=(10, 6))
print(coo_arr1.shape)
csr_arr1 = coo_arr1.tocsr()
coo_arr1.toarray()

In [ ]:
rows2 = np.random.randint(6, size=10)
cols2 = np.random.randint(6, size=10)
data2 = np.random.randint(10, size=10)

print(np.array([rows2, cols2, data2]))

coo_arr2 = sparse.coo_array((data2, (rows2, cols2)), shape=(10, 6))
csr_arr2 = coo_arr2.tocsr()
coo_arr2.toarray()

In [ ]:
(csr_arr1 @ csr_arr2.T).toarray()

In [ ]:
type(csr_arr1)

In [ ]:
type(sparse.vstack([coo_arr1, coo_arr2]))

In [ ]:
sparse.vstack([coo_arr1, coo_arr2]).toarray()

In [ ]:
csr_arr1[[0], :].toarray()

In [ ]:
seq = [csr_arr1] * 5
sparse.vstack(seq)

In [ ]:
rows3 = np.random.randint(6, size=10)
cols3 = np.random.randint(6, size=10)
data3 = np.random.randint(10, size=10)

print(np.array([rows3, cols3, data3]))

coo_mat = sparse.coo_matrix((data2, (rows2, cols2)), shape=(10, 6))
csr_mat = coo_mat.tocsr()
csr_mat.toarray()

In [ ]:
csr_mat

In [ ]:
A = np.ones((5, 5))
A

In [ ]:
S = sparse.csr_matrix(A)
S